# Neuro-Symbolic Pipeline Integration

This notebook combines:

1. TF-IDF V2 clinical-trial retrieval
2. Symbolic eligibility reasoning
3. Criterion-level proof traces
4. Missing-information handling
5. Explanation planning

The goal is to build one end-to-end pipeline from a synthetic patient profile to retrieved and symbolically evaluated clinical trials.

In [1]:
import json
from pathlib import Path

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
PROJECT_ROOT = Path("..")

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
PATIENTS_DIR = PROJECT_ROOT / "data" / "synthetic_patients"
RULES_DIR = PROJECT_ROOT / "data" / "rules"
RAG_DIR = PROJECT_ROOT / "data" / "rag"
SRC_DIR = PROJECT_ROOT / "src"

print("Project root:", PROJECT_ROOT.resolve())
print("Trials:", TRIALS_DIR.resolve())
print("Patients:", PATIENTS_DIR.resolve())
print("Rules:", RULES_DIR.resolve())
print("RAG:", RAG_DIR.resolve())
print("Source:", SRC_DIR.resolve())

Project root: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant
Trials: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\trials
Patients: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients
Rules: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules
RAG: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rag
Source: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\src


In [3]:
required_files = [
    TRIALS_DIR / "all_30_trials.csv",
    PATIENTS_DIR / "synthetic_patients.json",
    RULES_DIR / "patient_trial_rule_results.json",
    RULES_DIR / "patient_trial_evaluation_summary.csv",
    RULES_DIR / "missing_information_questions.csv",
    RAG_DIR / "tfidf_v2_benchmark_results.csv",
    SRC_DIR / "explanation_planner.py"
]

for path in required_files:
    print(
        path.name,
        "→",
        path.exists()
    )

all_30_trials.csv → True
synthetic_patients.json → True
patient_trial_rule_results.json → True
patient_trial_evaluation_summary.csv → True
missing_information_questions.csv → True
tfidf_v2_benchmark_results.csv → True
explanation_planner.py → True


In [4]:
trials_30 = pd.read_csv(
    TRIALS_DIR / "all_30_trials.csv"
)

print("Trials loaded:", len(trials_30))
print("Columns:", trials_30.columns.tolist())

trials_30.head(3)

Trials loaded: 30
Columns: ['Review Rank', 'NCT Number', 'Study Title', 'Study Status', 'Conditions', 'Age Categories', 'Study Type', 'Phase', 'Interventions', 'Last Update Posted', 'Study URL', 'Review Notes', 'Eligibility Text Retrieved', 'Minimum Age', 'Maximum Age', 'Full Eligibility Criteria']


,Review Rank,NCT Number,Study Title,Study Status,Conditions,Age Categories,Study Type,Phase,Interventions,Last Update Posted,Study URL,Review Notes,Eligibility Text Retrieved,Minimum Age,Maximum Age,Full Eligibility Criteria
0,1,NCT04616560,Trastuzumab Deruxtecan (DS-8201a) for the Trea...,RECRUITING,Desmoplastic Small Round Cell Tumor|Osteosarco...,"CHILD, ADULT",INTERVENTIONAL,PHASE2,PROCEDURE: Biospecimen Collection|PROCEDURE: C...,2026-09-29,https://clinicaltrials.gov/study/NCT04616560,Needs full criteria and minimum/maximum age re...,YES,12 Years,39 Years,Inclusion Criteria:\n\n* Phase 1 (Part A): Pat...
1,2,NCT05235165,Thoracotomy Versus Thoracoscopic Management of...,RECRUITING,Metastatic Malignant Neoplasm in the Lung|Meta...,"CHILD, ADULT",INTERVENTIONAL,PHASE3,PROCEDURE: Biospecimen Collection|PROCEDURE: C...,2026-08-25,https://clinicaltrials.gov/study/NCT05235165,Needs full criteria and minimum/maximum age re...,YES,NaN,50 Years,Inclusion Criteria:\n\n* Patients must be \< 5...
2,3,NCT06492954,Atezolizumab in Combination With Stereotactic ...,RECRUITING,Osteosarcoma|Pulmonary Recurrence of Osteosarcoma,"CHILD, ADULT",INTERVENTIONAL,PHASE1,PROCEDURE: Surgical Resection|RADIATION: Stere...,2026-09-30,https://clinicaltrials.gov/study/NCT06492954,Needs full criteria and minimum/maximum age re...,YES,NaN,50 Years,Inclusion Criteria:\n\n* Participants must hav...


In [5]:
with open(
    PATIENTS_DIR / "synthetic_patients.json",
    "r",
    encoding="utf-8"
) as file:
    synthetic_patients = json.load(file)

print("Patients loaded:", len(synthetic_patients))

synthetic_patients[0]

Patients loaded: 8


{'patient_id': 'P001',
 'age_years': 15,
 'sex': 'Male',
 'diagnosis': {'cancer_type': 'osteosarcoma',
  'histologic_confirmation': True,
  'grade': 'high-grade',
  'disease_status': 'newly_diagnosed',
  'disease_extent': 'localized',
  'metastatic_sites': [],
  'measurable_disease': True},
 'anthropometrics': {'weight_kg': 52, 'body_surface_area_m2': 1.55},
 'performance_status': {'scale': None, 'score': None},
 'prior_treatment': {'methotrexate': False,
  'doxorubicin': False,
  'cisplatin': False,
  'ifosfamide': False,
  'radiation': False,
  'major_surgery': False,
  'cellular_therapy': False,
  'investigational_agent': False},
 'current_treatment': {'map_chemotherapy': True,
  'methotrexate': True,
  'methotrexate_dose_g_m2': 12,
  'doxorubicin': True,
  'cisplatin': True,
  'ifosfamide': False,
  'chemotherapy_cycle': 1,
  'second_cycle_started': False},
 'treatment_timing_days': {'last_chemotherapy': None,
  'last_growth_factor': None,
  'last_radiation': None,
  'last_major_su

In [6]:
patients_by_id = {
    patient["patient_id"]: patient
    for patient in synthetic_patients
}

print(
    "Patient IDs:",
    list(patients_by_id.keys())
)

Patient IDs: ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008']


In [7]:
with open(
    RULES_DIR / "patient_trial_rule_results.json",
    "r",
    encoding="utf-8"
) as file:
    symbolic_results = json.load(file)

print(
    "Symbolic patient-trial evaluations:",
    len(symbolic_results)
)

symbolic_results[0]

Symbolic patient-trial evaluations: 40


{'patient_id': 'P001',
 'nct_id': 'NCT07560826',
 'overall_status': 'MET',
 'met_count': 18,
 'not_met_count': 0,
 'unknown_count': 0,
 'rule_results': [{'patient_id': 'P001',
   'rule_id': 'NCT07560826_R01',
   'criterion_type': 'inclusion',
   'field': 'age_years',
   'patient_value': 15,
   'operator': '<=',
   'expected_value': 22,
   'status': 'MET',
   'description': 'Participant must be 22 years of age or younger.'},
  {'patient_id': 'P001',
   'rule_id': 'NCT07560826_R02',
   'criterion_type': 'inclusion',
   'field': 'diagnosis.cancer_type',
   'patient_value': 'osteosarcoma',
   'operator': '==',
   'expected_value': 'osteosarcoma',
   'status': 'MET',
   'description': 'Participant must have osteosarcoma.'},
  {'patient_id': 'P001',
   'rule_id': 'NCT07560826_R03',
   'criterion_type': 'inclusion',
   'field': 'diagnosis.grade',
   'patient_value': 'high-grade',
   'operator': '==',
   'expected_value': 'high-grade',
   'status': 'MET',
   'description': 'Participant must ha

In [8]:
evaluation_summary_df = pd.read_csv(
    RULES_DIR / "patient_trial_evaluation_summary.csv"
)

print(
    "Evaluation summary rows:",
    len(evaluation_summary_df)
)

evaluation_summary_df.head()

Evaluation summary rows: 40


,patient_id,nct_id,overall_status,met_count,not_met_count,unknown_count
0,P001,NCT07560826,MET,18,0,0
1,P001,NCT07144254,NOT_MET,7,2,10
2,P001,NCT05590572,NOT_MET,15,1,7
3,P001,NCT06235125,UNKNOWN,8,0,6
4,P001,NCT02013336,NOT_MET,8,2,0


In [9]:
missing_questions_df = pd.read_csv(
    RULES_DIR / "missing_information_questions.csv"
)

print(
    "Missing-information rows:",
    len(missing_questions_df)
)

missing_questions_df.head()

Missing-information rows: 189


,patient_id,nct_id,rule_id,missing_field,question
0,P001,NCT07144254,NCT07144254_R05,performance_status.score,Additional information is needed for: performa...
1,P001,NCT07144254,NCT07144254_R06,treatment_timing_days.last_chemotherapy,When was the patient's most recent chemotherapy?
2,P001,NCT07144254,NCT07144254_R07,treatment_timing_days.last_major_surgery,Additional information is needed for: treatmen...
3,P001,NCT07144254,NCT07144254_R12,laboratory_values.alt_uln_ratio,What is the patient's most recent ALT value re...
4,P001,NCT07144254,NCT07144254_R13,pulmonary.dyspnea_at_rest,Additional information is needed for: pulmonar...


In [10]:
symbolic_trial_ids = sorted(
    evaluation_summary_df["nct_id"]
    .unique()
    .tolist()
)

print(
    "Trials with symbolic rules:",
    len(symbolic_trial_ids)
)

for nct_id in symbolic_trial_ids:
    print(nct_id)

Trials with symbolic rules: 5
NCT02013336
NCT05590572
NCT06235125
NCT07144254
NCT07560826


In [11]:
print("Trials:", len(trials_30))
print("Patients:", len(synthetic_patients))
print("Symbolic evaluations:", len(symbolic_results))
print("Symbolic trials:", len(symbolic_trial_ids))

Trials: 30
Patients: 8
Symbolic evaluations: 40
Symbolic trials: 5


In [12]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def build_trial_document(row):
    parts = [
        f"NCT ID: {clean_text(row['NCT Number'])}",
        f"Title: {clean_text(row['Study Title'])}",
        f"Status: {clean_text(row['Study Status'])}",
        f"Conditions: {clean_text(row['Conditions'])}",
        f"Age Categories: {clean_text(row['Age Categories'])}",
        f"Minimum Age: {clean_text(row['Minimum Age'])}",
        f"Maximum Age: {clean_text(row['Maximum Age'])}",
        f"Study Type: {clean_text(row['Study Type'])}",
        f"Phase: {clean_text(row['Phase'])}",
        f"Interventions: {clean_text(row['Interventions'])}",
        f"Eligibility Criteria: {clean_text(row['Full Eligibility Criteria'])}"
    ]

    return " ".join(parts)

In [13]:
trial_documents = trials_30.apply(
    build_trial_document,
    axis=1
).tolist()

print("Trial documents created:", len(trial_documents))

print("\nExample document:\n")
print(trial_documents[0][:1000])

Trial documents created: 30

Example document:

NCT ID: NCT04616560 Title: Trastuzumab Deruxtecan (DS-8201a) for the Treatment of Recurrent or Refractory Osteosarcoma, Wilms Tumor, and Desmoplastic Small Round Cell Tumor Status: RECRUITING Conditions: Desmoplastic Small Round Cell Tumor|Osteosarcoma|Recurrent Desmoplastic Small Round Cell Tumor|Recurrent Kidney Wilms Tumor|Recurrent Osteosarcoma|Refractory Desmoplastic Small Round Cell Tumor|Refractory Wilms Tumor|Wilms Tumor Age Categories: CHILD, ADULT Minimum Age: 12 Years Maximum Age: 39 Years Study Type: INTERVENTIONAL Phase: PHASE2 Interventions: PROCEDURE: Biospecimen Collection|PROCEDURE: Computed Tomography|PROCEDURE: Echocardiography Test|PROCEDURE: Magnetic Resonance Imaging|PROCEDURE: Multigated Acquisition Scan|BIOLOGICAL: Trastuzumab Deruxtecan Eligibility Criteria: Inclusion Criteria:

* Phase 1 (Part A): Patients must be at least 2 years and less than 12 years of age at the time of study enrollment
* Phase 2: Wilms tumo

In [14]:
def build_patient_query_v2(patient):

    diagnosis = patient["diagnosis"]
    prior = patient["prior_treatment"]
    current = patient["current_treatment"]
    timing = patient["treatment_timing_days"]
    labs = patient["laboratory_values"]
    cardiac = patient["cardiac"]
    pulmonary = patient["pulmonary"]
    clinical = patient["other_clinical_facts"]
    imaging = patient["surgical_imaging"]

    parts = []

    parts.append(
        f"{patient['age_years']} year old "
        f"{diagnosis['grade']} "
        f"{diagnosis['cancer_type']} patient."
    )

    parts.append(
        f"Disease is {diagnosis['disease_status']} "
        f"and {diagnosis['disease_extent']}."
    )

    if diagnosis["histologic_confirmation"] is True:
        parts.append(
            "Histologically confirmed osteosarcoma."
        )

    if diagnosis["measurable_disease"] is True:
        parts.append(
            "Measurable disease is present."
        )

    metastatic_sites = diagnosis.get(
        "metastatic_sites",
        []
    )

    if metastatic_sites:
        parts.append(
            "Metastatic disease sites include "
            + ", ".join(metastatic_sites)
            + "."
        )

    prior_drugs = []

    for drug in [
        "methotrexate",
        "doxorubicin",
        "cisplatin",
        "ifosfamide"
    ]:

        if prior.get(drug) is True:
            prior_drugs.append(drug)

    if prior_drugs:
        parts.append(
            "Prior anticancer treatment included "
            + ", ".join(prior_drugs)
            + "."
        )

    if current.get("map_chemotherapy") is True:
        parts.append(
            "Currently receiving or planning MAP chemotherapy."
        )

    if current.get("methotrexate_dose_g_m2") is not None:
        parts.append(
            f"Methotrexate dose "
            f"{current['methotrexate_dose_g_m2']} g/m2."
        )

    if timing["last_chemotherapy"] is not None:
        parts.append(
            f"Last chemotherapy was "
            f"{timing['last_chemotherapy']} days ago."
        )

    if timing["last_major_surgery"] is not None:
        parts.append(
            f"Last major surgery was "
            f"{timing['last_major_surgery']} days ago."
        )

    if clinical["life_expectancy_months"] is not None:
        parts.append(
            f"Life expectancy approximately "
            f"{clinical['life_expectancy_months']} months."
        )

    if clinical["blood_pressure_controlled"] is not None:
        parts.append(
            f"Blood pressure controlled: "
            f"{clinical['blood_pressure_controlled']}."
        )

    if labs["anc_per_ul"] is not None:
        parts.append(
            f"ANC {labs['anc_per_ul']} per uL."
        )

    if labs["platelets_per_ul"] is not None:
        parts.append(
            f"Platelets {labs['platelets_per_ul']} per uL."
        )

    if labs["hemoglobin_g_dl"] is not None:
        parts.append(
            f"Hemoglobin {labs['hemoglobin_g_dl']} g/dL."
        )

    if labs["gfr_ml_min_1_73m2"] is not None:
        parts.append(
            f"GFR {labs['gfr_ml_min_1_73m2']} "
            f"mL/min/1.73m2."
        )

    if cardiac["lvef_percent"] is not None:
        parts.append(
            f"LVEF {cardiac['lvef_percent']} percent."
        )

    if cardiac["qtc_ms"] is not None:
        parts.append(
            f"QTc {cardiac['qtc_ms']} milliseconds."
        )

    if pulmonary["oxygen_saturation_percent"] is not None:
        parts.append(
            f"Room air oxygen saturation "
            f"{pulmonary['oxygen_saturation_percent']} percent."
        )

    if imaging[
        "pulmonary_metastatic_disease_suspected"
    ] is True:
        parts.append(
            "Imaging suspicious for pulmonary metastatic disease."
        )

    if imaging[
        "largest_pulmonary_nodule_mm"
    ] is not None:
        parts.append(
            f"Pulmonary nodule measures "
            f"{imaging['largest_pulmonary_nodule_mm']} mm."
        )

    if imaging[
        "pulmonary_surgery_recommended"
    ] is True:
        parts.append(
            "Pulmonary surgery is recommended."
        )

    return " ".join(parts)

In [15]:
example_query = build_patient_query_v2(
    patients_by_id["P001"]
)

print(example_query)

15 year old high-grade osteosarcoma patient. Disease is newly_diagnosed and localized. Histologically confirmed osteosarcoma. Measurable disease is present. Currently receiving or planning MAP chemotherapy. Methotrexate dose 12 g/m2. ANC 1850 per uL. Platelets 168000 per uL. Hemoglobin 10.2 g/dL. GFR 96 mL/min/1.73m2. LVEF 61 percent.


In [16]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000
)

trial_matrix = tfidf_vectorizer.fit_transform(
    trial_documents
)

print(
    "TF-IDF matrix shape:",
    trial_matrix.shape
)

TF-IDF matrix shape: (30, 10000)


In [17]:
def retrieve_trials_v2(patient, top_k=5):

    query_text = build_patient_query_v2(patient)

    query_vector = tfidf_vectorizer.transform(
        [query_text]
    )

    similarity_scores = (
        query_vector @ trial_matrix.T
    ).toarray()[0]

    ranked_indices = (
        similarity_scores
        .argsort()[::-1][:top_k]
    )

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        row = trials_30.iloc[idx]

        results.append({
            "rank": rank,
            "nct_id": row["NCT Number"],
            "title": row["Study Title"],
            "study_status": row["Study Status"],
            "score": float(
                similarity_scores[idx]
            )
        })

    return pd.DataFrame(results)

In [18]:
retrieve_trials_v2(
    patients_by_id["P001"],
    top_k=5
)

,rank,nct_id,title,study_status,score
0,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,0.171073
1,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.123658
2,3,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.105753
3,4,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.089225
4,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.083688


In [19]:
expected_trial_by_patient = {
    "P001": "NCT07560826",
    "P003": "NCT07144254",
    "P005": "NCT05590572",
    "P006": "NCT06235125",
    "P007": "NCT02013336"
}

integration_retrieval_check = []

for patient_id, expected_trial in (
    expected_trial_by_patient.items()
):

    retrieved = retrieve_trials_v2(
        patients_by_id[patient_id],
        top_k=5
    )

    retrieved_ids = (
        retrieved["nct_id"]
        .tolist()
    )

    hit = (
        expected_trial in retrieved_ids
    )

    rank = (
        retrieved_ids.index(expected_trial) + 1
        if hit
        else None
    )

    integration_retrieval_check.append({
        "patient_id": patient_id,
        "expected_trial": expected_trial,
        "hit_at_5": hit,
        "rank": rank
    })

integration_retrieval_check_df = pd.DataFrame(
    integration_retrieval_check
)

integration_retrieval_check_df

,patient_id,expected_trial,hit_at_5,rank
0,P001,NCT07560826,True,1.0
1,P003,NCT07144254,True,3.0
2,P005,NCT05590572,True,2.0
3,P006,NCT06235125,True,5.0
4,P007,NCT02013336,False,NaN


In [20]:
symbolic_result_lookup = {}

for result in symbolic_results:
    key = (
        result["patient_id"],
        result["nct_id"]
    )

    symbolic_result_lookup[key] = result

print(
    "Symbolic result lookup entries:",
    len(symbolic_result_lookup)
)

Symbolic result lookup entries: 40


In [21]:
def get_symbolic_result(
    patient_id,
    nct_id
):
    return symbolic_result_lookup.get(
        (patient_id, nct_id)
    )

In [22]:
test_result = get_symbolic_result(
    "P001",
    "NCT07560826"
)

print(
    "Overall status:",
    test_result["overall_status"]
)

print(
    "Number of rules:",
    len(test_result["rule_results"])
)

Overall status: MET
Number of rules: 18


In [23]:
def retrieve_and_attach_symbolic(
    patient,
    top_k=5
):

    patient_id = patient["patient_id"]

    retrieved = retrieve_trials_v2(
        patient,
        top_k=top_k
    )

    integrated_results = []

    for _, row in retrieved.iterrows():

        nct_id = row["nct_id"]

        symbolic_result = get_symbolic_result(
            patient_id,
            nct_id
        )

        if symbolic_result is not None:

            symbolic_available = True
            symbolic_status = (
                symbolic_result["overall_status"]
            )

            met_count = sum(
                1
                for rule
                in symbolic_result["rule_results"]
                if rule["status"] == "MET"
            )

            not_met_count = sum(
                1
                for rule
                in symbolic_result["rule_results"]
                if rule["status"] == "NOT_MET"
            )

            unknown_count = sum(
                1
                for rule
                in symbolic_result["rule_results"]
                if rule["status"] == "UNKNOWN"
            )

        else:

            symbolic_available = False
            symbolic_status = None

            met_count = None
            not_met_count = None
            unknown_count = None

        integrated_results.append({
            "rank": row["rank"],
            "nct_id": nct_id,
            "title": row["title"],
            "retrieval_score": row["score"],
            "symbolic_available": symbolic_available,
            "symbolic_status": symbolic_status,
            "met_count": met_count,
            "not_met_count": not_met_count,
            "unknown_count": unknown_count
        })

    return pd.DataFrame(
        integrated_results
    )

In [24]:
p001_integrated = retrieve_and_attach_symbolic(
    patients_by_id["P001"],
    top_k=5
)

p001_integrated

,rank,nct_id,title,retrieval_score,symbolic_available,symbolic_status,met_count,not_met_count,unknown_count
0,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.171073,True,MET,18.0,0.0,0.0
1,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,0.123658,False,NaN,NaN,NaN,NaN
2,3,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,0.105753,False,NaN,NaN,NaN,NaN
3,4,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,0.089225,False,NaN,NaN,NaN,NaN
4,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,0.083688,True,NOT_MET,15.0,1.0,7.0


In [25]:
p004_integrated = retrieve_and_attach_symbolic(
    patients_by_id["P004"],
    top_k=10
)

p004_integrated

,rank,nct_id,title,retrieval_score,symbolic_available,symbolic_status,met_count,not_met_count,unknown_count
0,1,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,0.107812,False,NaN,NaN,NaN,NaN
1,2,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",0.105454,False,NaN,NaN,NaN,NaN
2,3,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.103408,True,UNKNOWN,15.0,0.0,4.0
3,4,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,0.102576,False,NaN,NaN,NaN,NaN
4,5,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,0.097015,False,NaN,NaN,NaN,NaN
5,6,NCT05235165,Thoracotomy Versus Thoracoscopic Management of...,0.092154,False,NaN,NaN,NaN,NaN
6,7,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,0.088810,False,NaN,NaN,NaN,NaN
7,8,NCT04616560,Trastuzumab Deruxtecan (DS-8201a) for the Trea...,0.084872,False,NaN,NaN,NaN,NaN
8,9,NCT03900793,Losartan + Sunitinib in Treatment of Osteosarcoma,0.082103,False,NaN,NaN,NaN,NaN
9,10,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,0.080935,False,NaN,NaN,NaN,NaN


In [26]:
p002_integrated = retrieve_and_attach_symbolic(
    patients_by_id["P002"],
    top_k=10
)

p002_integrated

,rank,nct_id,title,retrieval_score,symbolic_available,symbolic_status,met_count,not_met_count,unknown_count
0,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.160461,True,NOT_MET,14.0,4.0,0.0
1,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,0.151398,False,NaN,NaN,NaN,NaN
2,3,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",0.102581,False,NaN,NaN,NaN,NaN
3,4,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,0.097892,False,NaN,NaN,NaN,NaN
4,5,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",0.095352,False,NaN,NaN,NaN,NaN
5,6,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,0.086564,False,NaN,NaN,NaN,NaN
6,7,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,0.084015,False,NaN,NaN,NaN,NaN
7,8,NCT03628209,Nivolumab or Nivolumab and Azacitidine in Pati...,0.083281,False,NaN,NaN,NaN,NaN
8,9,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.082850,True,NOT_MET,5.0,4.0,10.0
9,10,NCT03900793,Losartan + Sunitinib in Treatment of Osteosarcoma,0.078751,False,NaN,NaN,NaN,NaN


In [27]:
p001_check = p001_integrated[
    p001_integrated["nct_id"] == "NCT07560826"
]

p001_check

,rank,nct_id,title,retrieval_score,symbolic_available,symbolic_status,met_count,not_met_count,unknown_count
0,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.171073,True,MET,18.0,0.0,0.0


In [28]:
p004_check = p004_integrated[
    p004_integrated["nct_id"] == "NCT07144254"
]

p004_check

,rank,nct_id,title,retrieval_score,symbolic_available,symbolic_status,met_count,not_met_count,unknown_count
2,3,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.103408,True,UNKNOWN,15.0,0.0,4.0


In [29]:
def split_rules_by_status(symbolic_result):

    grouped = {
        "MET": [],
        "NOT_MET": [],
        "UNKNOWN": []
    }

    for rule in symbolic_result["rule_results"]:
        grouped[rule["status"]].append(rule)

    return grouped

In [30]:
p002_symbolic = get_symbolic_result(
    "P002",
    "NCT07560826"
)

p002_grouped = split_rules_by_status(
    p002_symbolic
)

print("MET:", len(p002_grouped["MET"]))
print("NOT_MET:", len(p002_grouped["NOT_MET"]))
print("UNKNOWN:", len(p002_grouped["UNKNOWN"]))

MET: 14
NOT_MET: 4
UNKNOWN: 0


In [31]:
for rule in p002_grouped["NOT_MET"]:
    print("Rule:", rule["rule_id"])
    print("Description:", rule["description"])
    print("Patient value:", rule["patient_value"])
    print("Expected:", rule["operator"], rule["expected_value"])
    print("-" * 50)

Rule: NCT07560826_R12
Description: Absolute neutrophil count must be at least 1000/uL.
Patient value: 620
Expected: >= 1000
--------------------------------------------------
Rule: NCT07560826_R13
Description: Platelet count must be at least 100000/uL.
Patient value: 64000
Expected: >= 100000
--------------------------------------------------
Rule: NCT07560826_R14
Description: Hemoglobin must be at least 8.0 g/dL.
Patient value: 7.4
Expected: >= 8.0
--------------------------------------------------
Rule: NCT07560826_R15
Description: Left ventricular ejection fraction must be at least 50 percent.
Patient value: 43
Expected: >= 50
--------------------------------------------------


In [32]:
missing_questions_df.columns.tolist()

['patient_id', 'nct_id', 'rule_id', 'missing_field', 'question']

In [33]:
def get_missing_questions(
    patient_id,
    nct_id
):
    matches = missing_questions_df[
        (missing_questions_df["patient_id"] == patient_id)
        &
        (missing_questions_df["nct_id"] == nct_id)
    ]

    return matches[
        [
            "rule_id",
            "missing_field",
            "question"
        ]
    ].to_dict(
        orient="records"
    )

In [34]:
p004_questions = get_missing_questions(
    "P004",
    "NCT07144254"
)

print(
    "Questions found:",
    len(p004_questions)
)

for item in p004_questions:
    print(
        item["rule_id"],
        "→",
        item["question"]
    )

Questions found: 4
NCT07144254_R06 → When was the patient's most recent chemotherapy?
NCT07144254_R10 → What is the patient's current GFR or creatinine clearance?
NCT07144254_R12 → What is the patient's most recent ALT value relative to ULN?
NCT07144254_R17 → Is a recent QTc measurement available?


In [35]:
def build_proof_trace(
    patient_id,
    nct_id
):

    symbolic_result = get_symbolic_result(
        patient_id,
        nct_id
    )

    if symbolic_result is None:
        return None

    grouped = split_rules_by_status(
        symbolic_result
    )

    questions = get_missing_questions(
        patient_id,
        nct_id
    )

    return {
        "patient_id": patient_id,
        "nct_id": nct_id,
        "overall_status": symbolic_result[
            "overall_status"
        ],
        "met_rules": grouped["MET"],
        "not_met_rules": grouped["NOT_MET"],
        "unknown_rules": grouped["UNKNOWN"],
        "missing_questions": questions
    }

In [36]:
p004_proof = build_proof_trace(
    "P004",
    "NCT07144254"
)

print(
    "Overall status:",
    p004_proof["overall_status"]
)

print(
    "MET:",
    len(p004_proof["met_rules"])
)

print(
    "NOT_MET:",
    len(p004_proof["not_met_rules"])
)

print(
    "UNKNOWN:",
    len(p004_proof["unknown_rules"])
)

print(
    "Questions:",
    len(p004_proof["missing_questions"])
)

Overall status: UNKNOWN
MET: 15
NOT_MET: 0
UNKNOWN: 4
Questions: 4


In [37]:
for rule in p004_proof["unknown_rules"]:

    print("Rule:", rule["rule_id"])
    print("Description:", rule["description"])
    print("Patient value:", rule["patient_value"])
    print(
        "Expected:",
        rule["operator"],
        rule["expected_value"]
    )
    print("-" * 50)

Rule: NCT07144254_R06
Description: At least 14 days must have passed since myelosuppressive chemotherapy.
Patient value: None
Expected: >= 14
--------------------------------------------------
Rule: NCT07144254_R10
Description: Creatinine clearance or GFR must be at least 70 mL/min/1.73m2.
Patient value: None
Expected: >= 70
--------------------------------------------------
Rule: NCT07144254_R12
Description: ALT must be no more than 5 times ULN.
Patient value: None
Expected: <= 5
--------------------------------------------------
Rule: NCT07144254_R17
Description: QTc must be 470 ms or less.
Patient value: None
Expected: <= 470
--------------------------------------------------


In [38]:
for item in p004_proof[
    "missing_questions"
]:

    print(
        item["rule_id"],
        "→",
        item["question"]
    )

NCT07144254_R06 → When was the patient's most recent chemotherapy?
NCT07144254_R10 → What is the patient's current GFR or creatinine clearance?
NCT07144254_R12 → What is the patient's most recent ALT value relative to ULN?
NCT07144254_R17 → Is a recent QTc measurement available?


In [39]:
def run_integrated_pipeline(
    patient,
    top_k=5
):

    patient_id = patient["patient_id"]

    retrieved = retrieve_trials_v2(
        patient,
        top_k=top_k
    )

    integrated_results = []

    for _, row in retrieved.iterrows():

        nct_id = row["nct_id"]

        proof = build_proof_trace(
            patient_id,
            nct_id
        )

        if proof is not None:

            result = {
                "rank": int(row["rank"]),
                "nct_id": nct_id,
                "title": row["title"],
                "retrieval_score": float(
                    row["score"]
                ),
                "symbolic_available": True,
                "overall_status": proof[
                    "overall_status"
                ],
                "met_count": len(
                    proof["met_rules"]
                ),
                "not_met_count": len(
                    proof["not_met_rules"]
                ),
                "unknown_count": len(
                    proof["unknown_rules"]
                ),
                "proof_trace": proof,
                "missing_questions": proof[
                    "missing_questions"
                ]
            }

        else:

            result = {
                "rank": int(row["rank"]),
                "nct_id": nct_id,
                "title": row["title"],
                "retrieval_score": float(
                    row["score"]
                ),
                "symbolic_available": False,
                "overall_status": None,
                "met_count": None,
                "not_met_count": None,
                "unknown_count": None,
                "proof_trace": None,
                "missing_questions": []
            }

        integrated_results.append(result)

    return integrated_results

In [40]:
p001_pipeline = run_integrated_pipeline(
    patients_by_id["P001"],
    top_k=5
)

for result in p001_pipeline:

    print(
        result["rank"],
        result["nct_id"],
        result["symbolic_available"],
        result["overall_status"]
    )

1 NCT07560826 True MET
2 NCT07321912 False None
3 NCT03643133 False None
4 NCT06650514 False None
5 NCT05590572 True NOT_MET


In [41]:
p004_pipeline = run_integrated_pipeline(
    patients_by_id["P004"],
    top_k=5
)

for result in p004_pipeline:

    print(
        result["rank"],
        result["nct_id"],
        result["symbolic_available"],
        result["overall_status"],
        "questions:",
        len(result["missing_questions"])
    )

1 NCT03643133 False None questions: 0
2 NCT07725380 False None questions: 0
3 NCT07144254 True UNKNOWN questions: 4
4 NCT05019703 False None questions: 0
5 NCT07428993 False None questions: 0


In [42]:
p004_target = next(
    result
    for result in p004_pipeline
    if result["nct_id"] == "NCT07144254"
)

p004_target

{'rank': 3,
 'nct_id': 'NCT07144254',
 'title': 'Tegavivint With Gemcitabine in Patients With Relapsed or Refractory Osteosarcoma',
 'retrieval_score': 0.10340795788668289,
 'symbolic_available': True,
 'overall_status': 'UNKNOWN',
 'met_count': 15,
 'not_met_count': 0,
 'unknown_count': 4,
 'proof_trace': {'patient_id': 'P004',
  'nct_id': 'NCT07144254',
  'overall_status': 'UNKNOWN',
  'met_rules': [{'patient_id': 'P004',
    'rule_id': 'NCT07144254_R01',
    'criterion_type': 'inclusion',
    'field': 'diagnosis.histologic_confirmation',
    'patient_value': True,
    'operator': '==',
    'expected_value': True,
    'status': 'MET',
    'description': 'Osteosarcoma must have histologic verification.'},
   {'patient_id': 'P004',
    'rule_id': 'NCT07144254_R02',
    'criterion_type': 'inclusion',
    'field': 'diagnosis.disease_status',
    'patient_value': 'refractory',
    'operator': 'in',
    'expected_value': ['relapsed', 'refractory'],
    'status': 'MET',
    'description': '

In [43]:
def pipeline_summary_dataframe(
    pipeline_results
):

    rows = []

    for result in pipeline_results:

        rows.append({
            "rank": result["rank"],
            "nct_id": result["nct_id"],
            "title": result["title"],
            "retrieval_score": result[
                "retrieval_score"
            ],
            "symbolic_available": result[
                "symbolic_available"
            ],
            "overall_status": result[
                "overall_status"
            ],
            "met": result["met_count"],
            "not_met": result[
                "not_met_count"
            ],
            "unknown": result[
                "unknown_count"
            ],
            "missing_questions": len(
                result["missing_questions"]
            )
        })

    return pd.DataFrame(rows)

In [44]:
pipeline_summary_dataframe(
    p004_pipeline
)

,rank,nct_id,title,retrieval_score,symbolic_available,overall_status,met,not_met,unknown,missing_questions
0,1,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,0.107812,False,NaN,NaN,NaN,NaN,0
1,2,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",0.105454,False,NaN,NaN,NaN,NaN,0
2,3,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.103408,True,UNKNOWN,15.0,0.0,4.0,4
3,4,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,0.102576,False,NaN,NaN,NaN,NaN,0
4,5,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,0.097015,False,NaN,NaN,NaN,NaN,0


In [45]:
import sys

if str(SRC_DIR.resolve()) not in sys.path:
    sys.path.append(
        str(SRC_DIR.resolve())
    )

from explanation_planner import (
    build_explanation_plan,
    generate_missing_information_questions
)

print("Explanation planner imported successfully")

Explanation planner imported successfully


In [46]:
p004_symbolic_result = get_symbolic_result(
    "P004",
    "NCT07144254"
)

p004_explanation_plan = build_explanation_plan(
    p004_symbolic_result
)

p004_explanation_plan

{'patient_id': 'P004',
 'nct_id': 'NCT07144254',
 'overall_status': 'UNKNOWN',
 'conclusion': 'The available information is insufficient to determine whether all evaluated criteria are satisfied.',
 'decisive_rules': [{'rule_id': 'NCT07144254_R06',
   'description': 'At least 14 days must have passed since myelosuppressive chemotherapy.',
   'field': 'treatment_timing_days.last_chemotherapy',
   'patient_value': None,
   'operator': '>=',
   'expected_value': 14},
  {'rule_id': 'NCT07144254_R10',
   'description': 'Creatinine clearance or GFR must be at least 70 mL/min/1.73m2.',
   'field': 'laboratory_values.gfr_ml_min_1_73m2',
   'patient_value': None,
   'operator': '>=',
   'expected_value': 70},
  {'rule_id': 'NCT07144254_R12',
   'description': 'ALT must be no more than 5 times ULN.',
   'field': 'laboratory_values.alt_uln_ratio',
   'patient_value': None,
   'operator': '<=',
   'expected_value': 5},
  {'rule_id': 'NCT07144254_R17',
   'description': 'QTc must be 470 ms or less.

In [47]:
print(
    "Patient:",
    p004_explanation_plan["patient_id"]
)

print(
    "Trial:",
    p004_explanation_plan["nct_id"]
)

print(
    "Overall status:",
    p004_explanation_plan["overall_status"]
)

print(
    "Conclusion:",
    p004_explanation_plan["conclusion"]
)

print(
    "Requires more information:",
    p004_explanation_plan[
        "requires_more_information"
    ]
)

Patient: P004
Trial: NCT07144254
Overall status: UNKNOWN
Conclusion: The available information is insufficient to determine whether all evaluated criteria are satisfied.
Requires more information: True


In [48]:
p004_planner_questions = (
    generate_missing_information_questions(
        p004_explanation_plan
    )
)

print(
    "Questions generated:",
    len(p004_planner_questions)
)

for item in p004_planner_questions:
    print(
        item["rule_id"],
        "→",
        item["question"]
    )

Questions generated: 4
NCT07144254_R06 → How many days have passed since the patient's last chemotherapy?
NCT07144254_R10 → What is the patient's most recent GFR value in mL/min/1.73 m²?
NCT07144254_R12 → What is the patient's latest ALT value as a multiple of the upper limit of normal?
NCT07144254_R17 → What is the patient's most recent QTc measurement in milliseconds?


In [49]:
p002_symbolic_result = get_symbolic_result(
    "P002",
    "NCT07560826"
)

p002_explanation_plan = build_explanation_plan(
    p002_symbolic_result
)

print(
    "Overall status:",
    p002_explanation_plan[
        "overall_status"
    ]
)

print(
    "Conclusion:",
    p002_explanation_plan[
        "conclusion"
    ]
)

print(
    "Failed criteria:",
    len(
        p002_explanation_plan[
            "failed_rules"
        ]
    )
)

Overall status: NOT_MET
Conclusion: The patient does not satisfy all evaluated eligibility criteria.
Failed criteria: 4


In [50]:
for rule in p002_explanation_plan[
    "failed_rules"
]:

    print("Rule:", rule["rule_id"])
    print("Description:", rule["description"])
    print("Patient value:", rule["patient_value"])
    print(
        "Expected:",
        rule["operator"],
        rule["expected_value"]
    )
    print("-" * 50)

Rule: NCT07560826_R12
Description: Absolute neutrophil count must be at least 1000/uL.
Patient value: 620
Expected: >= 1000
--------------------------------------------------
Rule: NCT07560826_R13
Description: Platelet count must be at least 100000/uL.
Patient value: 64000
Expected: >= 100000
--------------------------------------------------
Rule: NCT07560826_R14
Description: Hemoglobin must be at least 8.0 g/dL.
Patient value: 7.4
Expected: >= 8.0
--------------------------------------------------
Rule: NCT07560826_R15
Description: Left ventricular ejection fraction must be at least 50 percent.
Patient value: 43
Expected: >= 50
--------------------------------------------------


In [51]:
p001_symbolic_result = get_symbolic_result(
    "P001",
    "NCT07560826"
)

p001_explanation_plan = build_explanation_plan(
    p001_symbolic_result
)

print(
    "Overall status:",
    p001_explanation_plan[
        "overall_status"
    ]
)

print(
    "Conclusion:",
    p001_explanation_plan[
        "conclusion"
    ]
)

print(
    "Requires more information:",
    p001_explanation_plan[
        "requires_more_information"
    ]
)

Overall status: MET
Conclusion: The patient satisfies all evaluated encoded criteria. This does not establish final clinical eligibility.
Requires more information: False


In [52]:
def run_neurosymbolic_pipeline(
    patient,
    top_k=5
):

    patient_id = patient["patient_id"]

    retrieved = retrieve_trials_v2(
        patient,
        top_k=top_k
    )

    final_results = []

    for _, row in retrieved.iterrows():

        nct_id = row["nct_id"]

        symbolic_result = get_symbolic_result(
            patient_id,
            nct_id
        )

        if symbolic_result is not None:

            proof = build_proof_trace(
                patient_id,
                nct_id
            )

            explanation_plan = (
                build_explanation_plan(
                    symbolic_result
                )
            )

            planner_questions = (
                generate_missing_information_questions(
                    explanation_plan
                )
            )

            result = {
                "rank": int(row["rank"]),
                "nct_id": nct_id,
                "title": row["title"],
                "retrieval_score": float(
                    row["score"]
                ),

                "symbolic_available": True,

                "overall_status":
                    symbolic_result[
                        "overall_status"
                    ],

                "met_count": len(
                    proof["met_rules"]
                ),

                "not_met_count": len(
                    proof["not_met_rules"]
                ),

                "unknown_count": len(
                    proof["unknown_rules"]
                ),

                "proof_trace": proof,

                "explanation_plan":
                    explanation_plan,

                "missing_questions":
                    planner_questions
            }

        else:

            result = {
                "rank": int(row["rank"]),
                "nct_id": nct_id,
                "title": row["title"],
                "retrieval_score": float(
                    row["score"]
                ),

                "symbolic_available": False,

                "overall_status": None,

                "met_count": None,
                "not_met_count": None,
                "unknown_count": None,

                "proof_trace": None,

                "explanation_plan": None,

                "missing_questions": []
            }

        final_results.append(result)

    return final_results

In [53]:
p001_final = run_neurosymbolic_pipeline(
    patients_by_id["P001"],
    top_k=5
)

for result in p001_final:

    print(
        result["rank"],
        result["nct_id"],
        result["symbolic_available"],
        result["overall_status"]
    )

1 NCT07560826 True MET
2 NCT07321912 False None
3 NCT03643133 False None
4 NCT06650514 False None
5 NCT05590572 True NOT_MET


In [54]:
p001_target = next(
    result
    for result in p001_final
    if result["nct_id"] == "NCT07560826"
)

print(
    "Status:",
    p001_target["overall_status"]
)

print(
    "Conclusion:",
    p001_target[
        "explanation_plan"
    ]["conclusion"]
)

print(
    "Missing questions:",
    len(
        p001_target[
            "missing_questions"
        ]
    )
)

Status: MET
Conclusion: The patient satisfies all evaluated encoded criteria. This does not establish final clinical eligibility.
Missing questions: 0


In [55]:
p004_final = run_neurosymbolic_pipeline(
    patients_by_id["P004"],
    top_k=5
)

In [56]:
p004_target = next(
    result
    for result in p004_final
    if result["nct_id"] == "NCT07144254"
)

In [57]:
print(
    "Status:",
    p004_target["overall_status"]
)

print(
    "Conclusion:",
    p004_target[
        "explanation_plan"
    ]["conclusion"]
)

print(
    "MET:",
    p004_target["met_count"]
)

print(
    "NOT_MET:",
    p004_target["not_met_count"]
)

print(
    "UNKNOWN:",
    p004_target["unknown_count"]
)

print(
    "Questions:",
    len(
        p004_target[
            "missing_questions"
        ]
    )
)

Status: UNKNOWN
Conclusion: The available information is insufficient to determine whether all evaluated criteria are satisfied.
MET: 15
NOT_MET: 0
UNKNOWN: 4
Questions: 4


In [58]:
for item in p004_target[
    "missing_questions"
]:

    print(
        item["rule_id"],
        "→",
        item["question"]
    )

NCT07144254_R06 → How many days have passed since the patient's last chemotherapy?
NCT07144254_R10 → What is the patient's most recent GFR value in mL/min/1.73 m²?
NCT07144254_R12 → What is the patient's latest ALT value as a multiple of the upper limit of normal?
NCT07144254_R17 → What is the patient's most recent QTc measurement in milliseconds?


In [59]:
p002_final = run_neurosymbolic_pipeline(
    patients_by_id["P002"],
    top_k=5
)

p002_target = next(
    result
    for result in p002_final
    if result["nct_id"] == "NCT07560826"
)

In [60]:
print(
    "Status:",
    p002_target["overall_status"]
)

print(
    "Conclusion:",
    p002_target[
        "explanation_plan"
    ]["conclusion"]
)

print(
    "MET:",
    p002_target["met_count"]
)

print(
    "NOT_MET:",
    p002_target["not_met_count"]
)

print(
    "UNKNOWN:",
    p002_target["unknown_count"]
)

Status: NOT_MET
Conclusion: The patient does not satisfy all evaluated eligibility criteria.
MET: 14
NOT_MET: 4
UNKNOWN: 0


In [61]:
def display_patient_results(
    patient,
    top_k=5
):

    patient_id = patient["patient_id"]

    results = run_neurosymbolic_pipeline(
        patient,
        top_k=top_k
    )

    print("=" * 80)
    print(
        f"PATIENT: {patient_id}"
    )
    print("=" * 80)

    for result in results:

        print()
        print(
            f"Rank {result['rank']} | "
            f"{result['nct_id']}"
        )

        print(
            f"Title: {result['title']}"
        )

        print(
            f"Retrieval score: "
            f"{result['retrieval_score']:.4f}"
        )

        if not result["symbolic_available"]:

            print(
                "Symbolic evaluation: "
                "Not available for this trial."
            )

            print(
                "This trial was retrieved as "
                "potentially relevant, but detailed "
                "symbolic eligibility rules have "
                "not yet been encoded."
            )

            print("-" * 80)

            continue

        print(
            f"Overall symbolic status: "
            f"{result['overall_status']}"
        )

        print(
            f"Criteria summary: "
            f"{result['met_count']} MET, "
            f"{result['not_met_count']} NOT_MET, "
            f"{result['unknown_count']} UNKNOWN"
        )

        print(
            "Conclusion:",
            result[
                "explanation_plan"
            ]["conclusion"]
        )

        if result["not_met_count"] > 0:

            print("\nFailed criteria:")

            for rule in result[
                "explanation_plan"
            ]["failed_rules"]:

                print(
                    f"- {rule['description']}"
                )

                print(
                    f"  Patient value: "
                    f"{rule['patient_value']}"
                )

                print(
                    f"  Required: "
                    f"{rule['operator']} "
                    f"{rule['expected_value']}"
                )

        if result["unknown_count"] > 0:

            print(
                "\nMissing-information questions:"
            )

            for item in result[
                "missing_questions"
            ]:

                print(
                    f"- {item['question']}"
                )

        print("-" * 80)

In [62]:
display_patient_results(
    patients_by_id["P001"],
    top_k=5
)

PATIENT: P001

Rank 1 | NCT07560826
Title: The CATSINDO Trial - Clinical and Translational Study in Newly Diagnosed Osteosarcoma
Retrieval score: 0.1711
Overall symbolic status: MET
Criteria summary: 18 MET, 0 NOT_MET, 0 UNKNOWN
Conclusion: The patient satisfies all evaluated encoded criteria. This does not establish final clinical eligibility.
--------------------------------------------------------------------------------

Rank 2 | NCT07321912
Title: Eflornithine (DFMO) for Ewing Sarcoma and Osteosarcoma
Retrieval score: 0.1237
Symbolic evaluation: Not available for this trial.
This trial was retrieved as potentially relevant, but detailed symbolic eligibility rules have not yet been encoded.
--------------------------------------------------------------------------------

Rank 3 | NCT03643133
Title: Mifamurtide Combined With Post-operative Chemotherapy for Newly Diagnosed High Risk Osteosarcoma Patients
Retrieval score: 0.1058
Symbolic evaluation: Not available for this trial.
This 

In [63]:
display_patient_results(
    patients_by_id["P002"],
    top_k=5
)

PATIENT: P002

Rank 1 | NCT07560826
Title: The CATSINDO Trial - Clinical and Translational Study in Newly Diagnosed Osteosarcoma
Retrieval score: 0.1605
Overall symbolic status: NOT_MET
Criteria summary: 14 MET, 4 NOT_MET, 0 UNKNOWN
Conclusion: The patient does not satisfy all evaluated eligibility criteria.

Failed criteria:
- Absolute neutrophil count must be at least 1000/uL.
  Patient value: 620
  Required: >= 1000
- Platelet count must be at least 100000/uL.
  Patient value: 64000
  Required: >= 100000
- Hemoglobin must be at least 8.0 g/dL.
  Patient value: 7.4
  Required: >= 8.0
- Left ventricular ejection fraction must be at least 50 percent.
  Patient value: 43
  Required: >= 50
--------------------------------------------------------------------------------

Rank 2 | NCT07321912
Title: Eflornithine (DFMO) for Ewing Sarcoma and Osteosarcoma
Retrieval score: 0.1514
Symbolic evaluation: Not available for this trial.
This trial was retrieved as potentially relevant, but detailed 

In [64]:
display_patient_results(
    patients_by_id["P004"],
    top_k=5
)

PATIENT: P004

Rank 1 | NCT03643133
Title: Mifamurtide Combined With Post-operative Chemotherapy for Newly Diagnosed High Risk Osteosarcoma Patients
Retrieval score: 0.1078
Symbolic evaluation: Not available for this trial.
This trial was retrieved as potentially relevant, but detailed symbolic eligibility rules have not yet been encoded.
--------------------------------------------------------------------------------

Rank 2 | NCT07725380
Title: Domatinostat With Sirolimus for Relapsed, Refractory Sarcoma and Osteosarcoma
Retrieval score: 0.1055
Symbolic evaluation: Not available for this trial.
This trial was retrieved as potentially relevant, but detailed symbolic eligibility rules have not yet been encoded.
--------------------------------------------------------------------------------

Rank 3 | NCT07144254
Title: Tegavivint With Gemcitabine in Patients With Relapsed or Refractory Osteosarcoma
Retrieval score: 0.1034
Overall symbolic status: UNKNOWN
Criteria summary: 15 MET, 0 NOT

In [65]:
display_patient_results(
    patients_by_id["P008"],
    top_k=5
)

PATIENT: P008

Rank 1 | NCT05235165
Title: Thoracotomy Versus Thoracoscopic Management of Pulmonary Metastases in Patients With Osteosarcoma
Retrieval score: 0.1338
Symbolic evaluation: Not available for this trial.
This trial was retrieved as potentially relevant, but detailed symbolic eligibility rules have not yet been encoded.
--------------------------------------------------------------------------------

Rank 2 | NCT07321912
Title: Eflornithine (DFMO) for Ewing Sarcoma and Osteosarcoma
Retrieval score: 0.1291
Symbolic evaluation: Not available for this trial.
This trial was retrieved as potentially relevant, but detailed symbolic eligibility rules have not yet been encoded.
--------------------------------------------------------------------------------

Rank 3 | NCT03643133
Title: Mifamurtide Combined With Post-operative Chemotherapy for Newly Diagnosed High Risk Osteosarcoma Patients
Retrieval score: 0.1268
Symbolic evaluation: Not available for this trial.
This trial was retr

In [66]:
INTEGRATED_DIR = PROJECT_ROOT / "data" / "integrated"

INTEGRATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Integrated output folder:",
    INTEGRATED_DIR.resolve()
)

Integrated output folder: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\integrated


In [67]:
all_integrated_results = []

for patient_id, patient in patients_by_id.items():

    results = run_neurosymbolic_pipeline(
        patient,
        top_k=5
    )

    all_integrated_results.append({
        "patient_id": patient_id,
        "results": results
    })

print(
    "Patients processed:",
    len(all_integrated_results)
)

Patients processed: 8


In [68]:
integrated_json_path = (
    INTEGRATED_DIR
    / "integrated_patient_trial_results.json"
)

with open(
    integrated_json_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        all_integrated_results,
        file,
        indent=2
    )

print(
    "Saved:",
    integrated_json_path.resolve()
)

Saved: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\integrated\integrated_patient_trial_results.json


In [69]:
summary_rows = []

for patient_entry in all_integrated_results:

    patient_id = patient_entry["patient_id"]

    for result in patient_entry["results"]:

        summary_rows.append({
            "patient_id": patient_id,
            "rank": result["rank"],
            "nct_id": result["nct_id"],
            "title": result["title"],
            "retrieval_score": result[
                "retrieval_score"
            ],
            "symbolic_available": result[
                "symbolic_available"
            ],
            "overall_status": result[
                "overall_status"
            ],
            "met_count": result[
                "met_count"
            ],
            "not_met_count": result[
                "not_met_count"
            ],
            "unknown_count": result[
                "unknown_count"
            ],
            "missing_questions_count": len(
                result[
                    "missing_questions"
                ]
            )
        })

integrated_summary_df = pd.DataFrame(
    summary_rows
)

integrated_summary_df.head(10)

,patient_id,rank,nct_id,title,retrieval_score,symbolic_available,overall_status,met_count,not_met_count,unknown_count,missing_questions_count
0,P001,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.171073,True,MET,18.0,0.0,0.0,0
1,P001,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,0.123658,False,NaN,NaN,NaN,NaN,0
2,P001,3,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,0.105753,False,NaN,NaN,NaN,NaN,0
3,P001,4,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,0.089225,False,NaN,NaN,NaN,NaN,0
4,P001,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,0.083688,True,NOT_MET,15.0,1.0,7.0,7
5,P002,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.160461,True,NOT_MET,14.0,4.0,0.0,0
6,P002,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,0.151398,False,NaN,NaN,NaN,NaN,0
7,P002,3,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",0.102581,False,NaN,NaN,NaN,NaN,0
8,P002,4,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,0.097892,False,NaN,NaN,NaN,NaN,0
9,P002,5,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",0.095352,False,NaN,NaN,NaN,NaN,0


In [70]:
print(
    "Integrated summary rows:",
    len(integrated_summary_df)
)

Integrated summary rows: 40


In [71]:
integrated_csv_path = (
    INTEGRATED_DIR
    / "integrated_patient_trial_summary.csv"
)

integrated_summary_df.to_csv(
    integrated_csv_path,
    index=False
)

print(
    "Saved:",
    integrated_csv_path.resolve()
)

Saved: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\integrated\integrated_patient_trial_summary.csv


In [72]:
symbolic_candidates_df = (
    integrated_summary_df[
        integrated_summary_df[
            "symbolic_available"
        ] == True
    ]
    .copy()
)

print(
    "Retrieved trials with symbolic rules:",
    len(symbolic_candidates_df)
)

symbolic_candidates_df

Retrieved trials with symbolic rules: 8


,patient_id,rank,nct_id,title,retrieval_score,symbolic_available,overall_status,met_count,not_met_count,unknown_count,missing_questions_count
0,P001,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.171073,True,MET,18.0,0.0,0.0,0
4,P001,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,0.083688,True,NOT_MET,15.0,1.0,7.0,7
5,P002,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,0.160461,True,NOT_MET,14.0,4.0,0.0,0
12,P003,3,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.114089,True,MET,19.0,0.0,0.0,0
17,P004,3,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.103408,True,UNKNOWN,15.0,0.0,4.0,4
21,P005,2,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,0.118295,True,MET,23.0,0.0,0.0,0
29,P006,5,NCT06235125,Safety and Feasibility of Intraoperative Visua...,0.101525,True,MET,14.0,0.0,0.0,0
38,P008,4,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,0.121568,True,NOT_MET,16.0,3.0,0.0,0


In [73]:
symbolic_candidates_path = (
    INTEGRATED_DIR
    / "retrieved_symbolic_candidates.csv"
)

symbolic_candidates_df.to_csv(
    symbolic_candidates_path,
    index=False
)

print(
    "Saved:",
    symbolic_candidates_path.resolve()
)

Saved: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\integrated\retrieved_symbolic_candidates.csv


In [74]:
output_files = [
    integrated_json_path,
    integrated_csv_path,
    symbolic_candidates_path
]

for path in output_files:
    print(
        path.name,
        "→",
        path.exists()
    )

integrated_patient_trial_results.json → True
integrated_patient_trial_summary.csv → True
retrieved_symbolic_candidates.csv → True


In [75]:
print("Patients:", len(patients_by_id))
print("Trials in retrieval corpus:", len(trials_30))
print("Symbolic evaluations:", len(symbolic_results))
print("Symbolic trials:", len(symbolic_trial_ids))
print("Integrated summary rows:", len(integrated_summary_df))

Patients: 8
Trials in retrieval corpus: 30
Symbolic evaluations: 40
Symbolic trials: 5
Integrated summary rows: 40


In [76]:
showcase_checks = [
    ("P001", "NCT07560826", "MET"),
    ("P002", "NCT07560826", "NOT_MET"),
    ("P004", "NCT07144254", "UNKNOWN")
]

for patient_id, nct_id, expected_status in showcase_checks:

    result = get_symbolic_result(
        patient_id,
        nct_id
    )

    actual_status = result["overall_status"]

    print(
        patient_id,
        nct_id,
        "→",
        actual_status,
        "| expected:",
        expected_status,
        "| pass:",
        actual_status == expected_status
    )

P001 NCT07560826 → MET | expected: MET | pass: True
P002 NCT07560826 → NOT_MET | expected: NOT_MET | pass: True
P004 NCT07144254 → UNKNOWN | expected: UNKNOWN | pass: True


In [77]:
print(
    f"TF-IDF V2 Hit@5: "
    f"{integration_retrieval_check_df['hit_at_5'].mean():.2f}"
)

TF-IDF V2 Hit@5: 0.80


In [78]:
for path in [
    integrated_json_path,
    integrated_csv_path,
    symbolic_candidates_path
]:
    print(
        path.name,
        "→",
        path.exists()
    )

integrated_patient_trial_results.json → True
integrated_patient_trial_summary.csv → True
retrieved_symbolic_candidates.csv → True


## Notebook 04 Conclusion

This notebook integrated the previously separate retrieval and symbolic reasoning components into one end-to-end neuro-symbolic pipeline.

Completed functionality:

- Loaded the 30-trial osteosarcoma retrieval corpus.
- Loaded 8 structured synthetic patient profiles.
- Reproduced the selected TF-IDF V2 retriever with Hit@5 = 0.80.
- Retrieved Top-5 candidate trials for each patient.
- Attached symbolic MET / NOT_MET / UNKNOWN results when encoded rules were available.
- Attached criterion-level proof traces.
- Generated missing-information questions for UNKNOWN criteria.
- Integrated the deterministic explanation planner.
- Produced readable patient-level outputs.
- Exported integrated JSON and CSV artifacts for future frontend use.

The next stage is constrained natural-language explanation generation and verification, followed by Streamlit integration.